# 03 — Phase 2: sub-pixel shoreline extraction

Thin Colab driver for `src/extract.py` (heavy logic lives in the module). Follows
the verification order in **PHASE2_SPEC.md §5**, anchored on the clean 2023
Sentinel-2 single scene (season 2022-2023) for the fetch → classify → extract →
visual-QC steps.

**Locked design (D1–D7):** scene-level extraction (one tide per scene); Series A
(38 annual products) + Series B (dense all-season 1999–2025); a LOCAL
sand/water/whitewater/other MLP per sensor group (TM/OLI/MSI); the water index is
a parameter (default MNDWI) with the Otsu/peaks threshold taken on **sand ∪ water
pixels only**; marching-squares sub-pixel contour; tidal-channel closure lines.

**Operator artifacts required before the classifier/extraction steps** (digitise
in QGIS, upload via the GitHub web interface — see the final cell):
`data/shoreline_search_zone.geojson`, `data/training_polygons.geojson`,
`data/reference_shorelines/`.


## Setup — clone-or-pull, authenticate GEE, import & reload `src/`

In [ ]:
import os
REPO_DIR = '/content/Shoreline-Dynamics'
# Clone-or-pull so Colab always runs the latest committed code.
if os.path.isdir(REPO_DIR):
    !cd {REPO_DIR} && git pull -q
else:
    !git clone https://github.com/SKPrince1911/Shoreline-Dynamics.git {REPO_DIR}
%cd {REPO_DIR}
!pip install -q -r requirements.txt

In [ ]:
import ee, geemap
ee.Authenticate()
ee.Initialize(project='shoreline-analysis')
print('GEE initialized:', ee.String('ok').getInfo())

In [ ]:
import sys; sys.path.append('.')
import importlib
import numpy as np, pandas as pd, geopandas as gpd
import matplotlib.pyplot as plt
from src import config, data, extract
# Pick up freshly pulled modules without a kernel restart (config before its
# dependents). extract is on the reload list so edits to src/extract.py land.
importlib.reload(config); importlib.reload(data); importlib.reload(extract)

aoi = ee.Geometry.Polygon(config.aoi_coordinates())
print('feature dim:', extract.feats_dim(),
      '| default index:', config.WATER_INDEX_DEFAULT,
      '| threshold:', config.THRESHOLD_METHOD_DEFAULT)
print('sensor groups:', extract.SENSOR_GROUP)

## 1. Scene lists — Series A (annual) and Series B (dense)

Series A explodes the approved inventory into one row per **acquisition date**
(same-overpass granules mosaicked; multi-date composites split), so every scene
keeps its own timestamp for Phase 3.

In [ ]:
annual = extract.build_scene_list_annual('outputs/image_inventory.csv')
n_years = annual['dry_year'].nunique()
print(f'Series A: {len(annual)} scenes across {n_years} dry-season-years')
assert n_years == 38, f'expected 38 non-gap years, got {n_years}'
assert annual['acq_datetime_utc'].notna().all(), 'found a NaT timestamp!'
print('sensors:', annual['sensor'].value_counts().to_dict())
annual.head(12)

Series B is a **large, all-season GEE query** (several hundred scenes,
chunked ≤8 aggregations per request to dodge HTTP 429). It writes
`outputs/scene_list_dense.csv`. Run it when you are ready to build the dense
series; it is not needed for the single-scene QC below.

In [ ]:
dense = extract.build_scene_list_dense()   # 1999-2025, L7/L8/L9/S2, all months
print(f'Series B: {len(dense)} scenes kept '
      f'(cloud <= {config.DENSE_CLOUD_MAX_PCT}%, coverage >= {config.DENSE_COVERAGE_MIN_PCT}%)')
print('per sensor:', dense['sensor'].value_counts().to_dict())
# sanity: SLC-off share from 2003, and the S2 ramp from ~2016
print('SLC-off L7 scenes:', int(dense['slc_off'].sum()))
dense.groupby('dry_year').size()

## 2. Fetch scenes — visual QC (RGB + valid mask)

Fetch is tiled onto a fixed **EPSG:32646** grid (pixel-aligned across scenes).
Before the search zone is digitised, pass a small `region_utm` window so 10 m
Sentinel-2 stays within Colab RAM. Here we use a ~6 km box over Kolatoli /
Marine Drive.

In [ ]:
from shapely.geometry import box
demo_ll = box(92.00, 21.33, 92.06, 21.42)     # (minlon, minlat, maxlon, maxlat)
demo_utm = extract._to_utm(demo_ll)

def rgb_from_scene(sc):
    rgb = np.dstack([sc.bands['red'], sc.bands['green'], sc.bands['blue']])
    return np.clip(rgb / 0.3, 0, 1)           # display stretch 0..0.3 reflectance

def show_scene(sc, title=''):
    fig, ax = plt.subplots(1, 2, figsize=(13, 6))
    ax[0].imshow(rgb_from_scene(sc)); ax[0].set_title(title + ' — RGB'); ax[0].axis('off')
    ax[1].imshow(sc.valid, cmap='gray'); ax[1].set_title('valid mask'); ax[1].axis('off')
    plt.tight_layout(); plt.show()
    gmin = float(np.nanmin(sc.bands['green'])); gmax = float(np.nanmax(sc.bands['green']))
    print(title, sc.sensor, sc.image_id[:44], '| acq', sc.acq_datetime_utc,
          '| georef', sc.georef_rmse_m, 'm | green range [%.3f, %.3f]' % (gmin, gmax),
          '| valid %.1f%%' % (100 * sc.valid.mean()))
    assert -0.2 <= gmin and gmax <= 1.6, 'reflectance outside the expected [0,1] range'

def row_for(df, sensor=None, dry_year=None):
    q = df
    if sensor is not None: q = q[q['sensor'] == sensor]
    if dry_year is not None: q = q[q['dry_year'] == dry_year]
    return q.iloc[0]

In [ ]:
# The 2023 (2022-2023) clean single Sentinel-2 product.
s2_row = row_for(annual, sensor='S2', dry_year=2023)
sc_s2 = extract.fetch_scene(s2_row, region_utm=demo_utm)
show_scene(sc_s2, '2023 (2022-2023)')

In [ ]:
# One Landsat-5 and one Landsat-8 scene, same window, to eyeball cross-sensor look.
sc_l5 = extract.fetch_scene(row_for(annual, sensor='L5'), region_utm=demo_utm)
show_scene(sc_l5, 'Landsat-5 sample')
sc_l8 = extract.fetch_scene(row_for(annual, sensor='L8'), region_utm=demo_utm)
show_scene(sc_l8, 'Landsat-8 sample')

## 3. Train the Sentinel-2 (MSI) classifier

**Requires `data/training_polygons.geojson`** (class ∈ {other, sand, whitewater,
water}). For a real run also digitise `data/shoreline_search_zone.geojson` first,
so training scenes are fetched over the whole coastal band and every polygon
contributes. **Gate:** per-class precision/recall ≥ 0.90 for *water* and *sand*;
if not, add training polygons and re-run.

In [ ]:
TRAIN_PATH = 'data/training_polygons.geojson'
clf_msi = None
if not os.path.exists(TRAIN_PATH):
    print('BLOCKED:', TRAIN_PATH, 'not found.')
    print('Digitise it in QGIS (see the final cell), upload via the GitHub web')
    print('interface, then `git pull` in the setup cell and re-run.')
else:
    tp = gpd.read_file(TRAIN_PATH)
    print('label polygons by class:', tp['class'].value_counts().to_dict())
    # Fetch over the full search zone if present, else the demo window.
    region = None if os.path.exists(config.SEARCH_ZONE_PATH) else demo_utm
    train_rows = annual[annual['sensor'] == 'S2'].head(5)
    scenes = [extract.fetch_scene(r, region_utm=region) for _, r in train_rows.iterrows()]
    fit_scenes, hold = scenes[:-1], scenes[-1]     # hold out one scene (different date)
    X, y = extract.build_training_set(fit_scenes, TRAIN_PATH)
    uniq, cnt = np.unique(y, return_counts=True)
    print('training samples:', X.shape, '| per class:', dict(zip(uniq.tolist(), cnt.tolist())))
    clf_msi = extract.train_classifier(X, y, 'MSI')
    Xh, yh = extract.build_training_set([hold], TRAIN_PATH)
    print(extract.evaluate_classifier(clf_msi, Xh, yh, write_report=True, sensor_group='MSI'))

## 4. Extract the 2023 shoreline + visual QC

Classify → MNDWI → Otsu on the **sand ∪ water interface** → marching-squares
sub-pixel contour → clip to the search zone and follow the tidal-channel closure
lines. Visually: the red contour should track the wet/dry land–water boundary,
sit off the Marine Drive road edge, not run up the tidal channels (cyan), and
not hug the cloud-mask boundary.

In [ ]:
if not os.path.exists(TRAIN_PATH):
    print('BLOCKED: train the MSI classifier first (needs data/training_polygons.geojson).')
    line = None
else:
    settings = extract.default_settings(water_index_name='mndwi', threshold_method='otsu')
    # Full extraction region if the search zone exists, else the demo window.
    if os.path.exists(config.SEARCH_ZONE_PATH):
        sc = extract.fetch_scene(s2_row)
    else:
        sc = sc_s2
        print('note: no search zone yet -> extracting within the demo window only.')
    clf = clf_msi if clf_msi is not None else extract.load_classifier('MSI')
    labels = extract.classify_scene(sc, clf)
    idx = extract.water_index(sc, 'mndwi')
    thr = extract.interface_threshold(idx, labels, 'otsu')
    contours = extract.extract_contour(idx, thr, sc.valid, sc.transform)
    line = extract.filter_contours(contours, settings['search_zone'], settings['channel_lines'])
    print('interface Otsu threshold (sand-union-water only):', round(float(thr), 4))
    print('raw contour segments:', len(contours),
          '| final shoreline length_m:', None if line is None else round(line.length, 1))

    t, (H, W) = sc.transform, sc.valid.shape
    extent = [t.c, t.c + W * t.a, t.f + H * t.e, t.f]   # [xmin, xmax, ymin, ymax] UTM
    overlay = np.zeros((H, W, 3))
    overlay[labels == config.CLASS_SAND] = [0.96, 0.88, 0.30]
    overlay[labels == config.CLASS_WATER] = [0.17, 0.50, 0.72]
    overlay[labels == config.CLASS_WHITEWATER] = [1.0, 1.0, 1.0]
    fig, ax = plt.subplots(1, 3, figsize=(17, 6))
    ax[0].imshow(rgb_from_scene(sc)); ax[0].set_title('RGB'); ax[0].axis('off')
    ax[1].imshow(overlay); ax[1].set_title('classified sand / water / whitewater'); ax[1].axis('off')
    ax[2].imshow(rgb_from_scene(sc), extent=extent, origin='upper')
    if line is not None:
        for ln in extract._as_line_list(line):
            xs, ys = ln.xy; ax[2].plot(xs, ys, '-', color='red', lw=1.2)
    for cl in settings['channel_lines']:
        xs, ys = cl.xy; ax[2].plot(xs, ys, color='cyan', lw=2)
    ax[2].set_aspect('equal'); ax[2].set_title('sub-pixel shoreline (red) + channels (cyan)')
    plt.tight_layout(); plt.show()

## 5. Zoomed comparison — how tightly the contour follows the boundary

In [ ]:
if 'line' in globals() and line is not None:
    mid = line.interpolate(0.5, normalized=True)
    half = 750  # metres
    fig, ax = plt.subplots(figsize=(9, 9))
    ax.imshow(rgb_from_scene(sc), extent=extent, origin='upper')
    for ln in extract._as_line_list(line):
        xs, ys = ln.xy; ax.plot(xs, ys, '-', color='red', lw=1.6)
    ax.set_xlim(mid.x - half, mid.x + half); ax.set_ylim(mid.y - half, mid.y + half)
    ax.set_aspect('equal'); ax.set_title('zoom: sub-pixel contour vs land/water boundary')
    plt.show()
else:
    print('Run the extraction cell first (needs the trained classifier).')

## 6. Batch extraction — Series A, then Series B

Run only once the single-scene QC looks right **and** a classifier exists for
every sensor group the list needs (Series A spans TM/OLI/MSI). `extract_all`
checkpoints to `outputs/shorelines/sds_scenes.geojson` after **every** scene and
resumes from it, so a Colab disconnect never costs the whole run.

In [ ]:
settings = extract.default_settings()
gdfA = extract.extract_all(annual, settings)             # Series A (~60 scenes)
merged = extract.merge_annual(gdfA)                      # per-segment source for Phase 3
os.makedirs(extract.SHORELINE_DIR, exist_ok=True)
merged.to_file(extract.SDS_ANNUAL_PATH, driver='GeoJSON')
print('Series A scenes extracted:', len(gdfA), '-> merged segments:', len(merged))

# Series B (hundreds of scenes) — uncomment when ready:
# gdfB = extract.extract_all(dense, settings)
# Push outputs/ to Drive + GitHub with the save_outputs() cell from notebook 02.

## Operator artifacts to digitise in QGIS (upload via the GitHub web interface)

These are the manual inputs the extraction reads. Upload them under `data/` on
GitHub, then `git pull` in the setup cell.

**1. `data/shoreline_search_zone.geojson`** — one polygon (EPSG:4326) enclosing
every plausible shoreline position 1988–2025: roughly ±300 m about the present
shoreline along the open Marine Drive coast, widening to ±800–1000 m within ~3 km
of the Bakkhali and Naf river mouths. Replaces CoastSat's scalar `max_dist_ref`
and keeps extraction human-controlled and auditable.

**2. `data/training_polygons.geojson`** — labelled polygons with a `class` field
in {`other`, `sand`, `whitewater`, `water`}. Digitise ~6 scenes per sensor group
(TM = L4/L5/L7, OLI = L8/L9, MSI = S2), spread across dry/monsoon and along the
AOI (Bakkhali mouth, Kolatoli/Marine Drive, Himchari cliffs, Inani, Teknaf, Shah
Porir Dwip). Aim for ≥5,000 labelled pixels per class per sensor group.

**3. `data/reference_shorelines/`** — 10 manually digitised validation
shorelines (LineStrings, EPSG:4326) with an `image_id` property matching the
scene, digitised at native resolution (~1:2,000) along the wet/dry line. Used by
`benchmark_extraction()` to pick the operational index/threshold and by
`intersensor_bias()`.

Generated outputs (`outputs/scene_list_dense.csv`,
`outputs/shorelines/*.geojson`, `outputs/extraction_log.csv`, `models/*.joblib`)
are pushed to Drive + GitHub by the existing `save_outputs()` — no manual upload.
